In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style="color:red"> 코로나 기간 정상화별 학습모델링 최종</span>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import time

# 시계열 모델 라이브러리
try:
    from prophet import Prophet
    prophet_available = True
    print("Prophet 사용 가능")
except ImportError:
    prophet_available = False
    print("Prophet 설치 필요: pip install prophet")

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.exponential_smoothing.ets import ETSModel
    statsmodels_available = True
    print("Statsmodels 사용 가능")
except ImportError:
    statsmodels_available = False
    print("Statsmodels 설치 필요: pip install statsmodels")

# 기본 설정
warnings.filterwarnings('ignore')
plt.style.use('default')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

# 한글 폰트 설정 (Windows/Mac/Linux 호환)
try:
    import matplotlib.font_manager as fm
    font_list = [f.name for f in fm.fontManager.ttflist]
    
    if 'Malgun Gothic' in font_list:
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif 'AppleGothic' in font_list:
        plt.rcParams['font.family'] = 'AppleGothic'
    elif 'NanumGothic' in font_list:
        plt.rcParams['font.family'] = 'NanumGothic'
    else:
        plt.rcParams['font.family'] = 'DejaVu Sans'
        print("한글 폰트 없음. 영문으로 표시됩니다.")
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로딩 완료")
print(f"작업 시작 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Prophet 사용 가능
Statsmodels 사용 가능
라이브러리 로딩 완료
작업 시작 시간: 2025-07-21 17:43:49


In [3]:
# 데이터 파일 경로 및 기준 설정
FILE_PATH = 'C:/ai_x2/source/proz2/외국인입국자_전처리완료_딥러닝용.csv'
LAST_DATA_YEAR = 2025
LAST_DATA_MONTH = 5
PREDICTION_START_YEAR = 2025
PREDICTION_START_MONTH = 6
MAX_PREDICTION_YEAR = 2030

def load_and_preprocess_data(file_path):
    """외국인 입국자 데이터 로드 및 기본 전처리"""
    df = pd.read_csv(file_path)
    
    # 날짜 컬럼 생성
    df['date'] = pd.to_datetime(df['연도'].astype(str) + '-' + df['월'].astype(str).str.zfill(2))
    
    # 카테고리 인코딩
    le_nationality = LabelEncoder()
    le_purpose = LabelEncoder()
    df['국적_encoded'] = le_nationality.fit_transform(df['국적'])
    df['목적_encoded'] = le_purpose.fit_transform(df['목적'])
    
    # 계절 인코딩 (순환 특성 반영)
    season_mapping = {'봄': 0, '여름': 1, '가을': 2, '겨울': 3}
    df['계절_encoded'] = df['계절'].map(season_mapping)
    df['계절_sin'] = np.sin(2 * np.pi * df['계절_encoded'] / 4)
    df['계절_cos'] = np.cos(2 * np.pi * df['계절_encoded'] / 4)
    
    # 월 순환 특성
    df['월_sin'] = np.sin(2 * np.pi * df['월'] / 12)
    df['월_cos'] = np.cos(2 * np.pi * df['월'] / 12)
    
    return df, le_nationality, le_purpose

def trend_correction_normalization(df):
    """방법 1: 트렌드 보정"""
    df_corrected = df.copy()
    
    for (nationality, purpose), group in df.groupby(['국적', '목적']):
        # 2005-2019년 데이터로 트렌드 계산
        normal_period = group[group['코로나기간'] == 0]
        covid_period = group[group['코로나기간'] == 1]
        
        if len(normal_period) > 12 and len(covid_period) > 0:
            # 2019년 마지막 12개월 평균을 기준으로 설정
            baseline_2019 = normal_period[normal_period['연도'] == 2019]['입국자수'].mean()
            
            if baseline_2019 > 0:
                # 2005-2019년 연평균 성장률 계산
                yearly_avg = normal_period.groupby('연도')['입국자수'].mean()
                if len(yearly_avg) > 1:
                    growth_rate = (yearly_avg.iloc[-1] / yearly_avg.iloc[0]) ** (1/(len(yearly_avg)-1)) - 1
                    growth_rate = max(-0.1, min(0.1, growth_rate))  # -10%~+10% 제한
                else:
                    growth_rate = 0
                
                # 코로나 이후 데이터 정상화 (2024년부터 정상 회귀 가정)
                for idx, row in covid_period.iterrows():
                    if row['연도'] >= 2024:
                        years_from_2019 = row['연도'] - 2019
                        expected_normal = baseline_2019 * ((1 + growth_rate) ** years_from_2019)
                        
                        # 계절성 반영 (2019년 동월 대비 비율)
                        month_data_2019 = normal_period[(normal_period['연도'] == 2019) & (normal_period['월'] == row['월'])]
                        if len(month_data_2019) > 0 and baseline_2019 > 0:
                            seasonal_factor = month_data_2019['입국자수'].iloc[0] / baseline_2019
                            df_corrected.loc[idx, '입국자수'] = expected_normal * seasonal_factor
    
    df_corrected['normalization_method'] = 1
    return df_corrected

def structural_change_normalization(df):
    """방법 2: 구조적 변화 모델링"""
    df_structural = df.copy()
    
    # 코로나 기간에 구조 변화 변수 추가
    df_structural['regime'] = np.where(df_structural['코로나기간'] == 1, 'covid', 'normal')
    
    # 회복 기간 변수 추가 (2024년부터 점진적 회복)
    df_structural['recovery_factor'] = 1.0
    
    for idx, row in df_structural.iterrows():
        if row['연도'] == 2024:
            df_structural.loc[idx, 'recovery_factor'] = 0.7  # 70% 회복
        elif row['연도'] == 2025:
            df_structural.loc[idx, 'recovery_factor'] = 0.9  # 90% 회복
        elif row['연도'] >= 2026:
            df_structural.loc[idx, 'recovery_factor'] = 1.0  # 완전 회복
    
    df_structural['normalization_method'] = 2
    return df_structural

def extrapolation_normalization(df):
    """방법 3: 외삽법"""
    df_extrapolated = df.copy()
    
    for (nationality, purpose), group in df.groupby(['국적', '목적']):
        normal_period = group[group['코로나기간'] == 0]
        covid_period = group[group['코로나기간'] == 1]
        
        if len(normal_period) > 24:  # 최소 2년 데이터 필요
            # 연도별 평균으로 트렌드 계산
            yearly_avg = normal_period.groupby('연도')['입국자수'].mean()
            monthly_pattern = normal_period.groupby('월')['입국자수'].mean()
            
            if len(yearly_avg) > 1 and monthly_pattern.sum() > 0:
                # 트렌드 연장
                for idx, row in covid_period.iterrows():
                    if row['연도'] >= 2020:
                        # 마지막 정상년도(2019) 기준 연장
                        last_normal_year = yearly_avg.index[-1]
                        last_normal_value = yearly_avg.iloc[-1]
                        
                        # 월별 계절성 적용
                        if row['월'] in monthly_pattern.index:
                            seasonal_factor = monthly_pattern[row['월']] / yearly_avg.mean()
                            extrapolated_value = last_normal_value * seasonal_factor
                            df_extrapolated.loc[idx, '입국자수'] = max(0, extrapolated_value)
    
    df_extrapolated['normalization_method'] = 3
    return df_extrapolated

def bridge_normalization(df):
    """방법 5: 브리지 모델링"""
    df_bridge = df.copy()
    
    for (nationality, purpose), group in df.groupby(['국적', '목적']):
        # 앵커 포인트 설정
        baseline_2019 = group[group['연도'] == 2019]['입국자수'].mean()
        baseline_2024 = group[group['연도'] == 2024]['입국자수'].mean()
        
        if pd.notna(baseline_2019) and pd.notna(baseline_2024) and baseline_2019 > 0:
            # 2020-2023년 구간을 브리지 곡선으로 대체
            covid_period = group[(group['연도'] >= 2020) & (group['연도'] <= 2023)]
            
            for idx, row in covid_period.iterrows():
                # 진행률 계산 (2020-2023년을 0-1로 매핑)
                progress = (row['연도'] - 2020) / 4 + (row['월'] - 1) / (4 * 12)
                progress = min(1.0, max(0.0, progress))
                
                # S-커브 적용 (지수 회복)
                recovery_factor = 1 - np.exp(-3 * progress)
                
                # 브리지 값 계산
                bridge_value = baseline_2019 + (baseline_2024 - baseline_2019) * recovery_factor
                
                # 계절성 반영
                month_data_2019 = group[(group['연도'] == 2019) & (group['월'] == row['월'])]
                if len(month_data_2019) > 0 and baseline_2019 > 0:
                    seasonal_factor = month_data_2019['입국자수'].iloc[0] / baseline_2019
                    df_bridge.loc[idx, '입국자수'] = max(0, bridge_value * seasonal_factor)
    
    df_bridge['normalization_method'] = 5
    return df_bridge

def apply_normalization(df, method):
    """6가지 정상화 방법 적용"""
    print(f"정상화 방법 {method} 적용 중...")
    
    if method == 0:
        # 방법 0: 코로나 기간 제외
        df_normalized = df.copy()
        df_normalized['normalization_method'] = 0
        
    elif method == 1:
        # 방법 1: 트렌드 보정
        df_normalized = trend_correction_normalization(df)
        
    elif method == 2:
        # 방법 2: 구조적 변화 모델링
        df_normalized = structural_change_normalization(df)
        
    elif method == 3:
        # 방법 3: 외삽법
        df_normalized = extrapolation_normalization(df)
        
    elif method == 4:
        # 방법 4: 코로나 포함 전체
        df_normalized = df.copy()
        df_normalized['normalization_method'] = 4
        
    elif method == 5:
        # 방법 5: 브리지 모델링
        df_normalized = bridge_normalization(df)
        
    else:
        raise ValueError("정상화 방법은 0~5 중 하나여야 합니다.")
    
    return df_normalized

# 데이터 로드
print("데이터 로딩 중...")
df_original, le_nationality, le_purpose = load_and_preprocess_data(FILE_PATH)

print(f"데이터 로드 완료: {df_original.shape}")

min_date_str = df_original['date'].min().strftime('%Y-%m-%d')
max_date_str = df_original['date'].max().strftime('%Y-%m-%d')

print(f"날짜 범위: {min_date_str} ~ {max_date_str}")
print(f"국적 수: {df_original['국적'].nunique()}, 목적 수: {df_original['목적'].nunique()}, 계절 수: {df_original['계절'].nunique()}")

데이터 로딩 중...
데이터 로드 완료: (59780, 23)
날짜 범위: 2005-01-01 ~ 2025-05-01
국적 수: 61, 목적 수: 4, 계절 수: 4


In [4]:
def get_user_input():
    """사용자로부터 분석 조건 입력받기"""
    print("="*60)
    print("외국인 입국자 예측 분석")
    print("="*60)
    print(f"현재 데이터: {LAST_DATA_YEAR}년 {LAST_DATA_MONTH}월까지")
    print()
    
    # 1. 정상화 방법 선택
    normalization_methods = {
        0: "코로나 기간 제외",
        1: "트렌드 보정", 
        2: "구조적 변화 모델링",
        3: "외삽법",
        4: "코로나 포함 전체",
        5: "브리지 모델링"
    }
    
    print("정상화 방법:")
    for key, value in normalization_methods.items():
        print(f"  {key}: {value}")
    
    while True:
        try:
            normalization = int(input("정상화 방법 선택 (0-5): "))
            if 0 <= normalization <= 5:
                print(f"선택됨: {normalization_methods[normalization]}")
                break
            else:
                print("0~5 사이의 숫자를 입력하세요.")
        except ValueError:
            print("올바른 숫자를 입력하세요.")
    
    # 2. 국적 선택
    available_nationalities = sorted(df_original['국적'].unique())
    print(f"\n사용 가능한 국적: {len(available_nationalities)}개")
    print("예시:", ', '.join(available_nationalities[:5]), "...")
    
    nationality_input = input("국적 선택 (엔터=전체): ").strip()
    if nationality_input == "":
        selected_nationality = "전체"
        print("선택됨: 전체 국적")
    elif nationality_input in available_nationalities:
        selected_nationality = nationality_input
        print(f"선택됨: {nationality_input}")
    else:
        print(f"'{nationality_input}'는 존재하지 않습니다. 전체로 설정합니다.")
        selected_nationality = "전체"
    
    # 3. 목적 선택
    available_purposes = sorted(df_original['목적'].unique())
    print(f"\n사용 가능한 목적: {', '.join(available_purposes)}")
    
    purpose_input = input("목적 선택 (엔터=전체): ").strip()
    if purpose_input == "":
        selected_purpose = "전체"
        print("선택됨: 전체 목적")
    elif purpose_input in available_purposes:
        selected_purpose = purpose_input
        print(f"선택됨: {purpose_input}")
    else:
        print(f"'{purpose_input}'는 존재하지 않습니다. 전체로 설정합니다.")
        selected_purpose = "전체"
    
    # 4. 계절 선택
    available_seasons = ['봄', '여름', '가을', '겨울']
    print(f"\n사용 가능한 계절: {', '.join(available_seasons)}")
    
    season_input = input("계절 선택 (엔터=전체): ").strip()
    if season_input == "":
        selected_season = "전체"
        print("선택됨: 전체 계절")
    elif season_input in available_seasons:
        selected_season = season_input
        print(f"선택됨: {season_input}")
    else:
        print(f"'{season_input}'는 존재하지 않습니다. 전체로 설정합니다.")
        selected_season = "전체"
    
    # 5. 예측 기간 설정 (새로운 방식)
    print("\n" + "="*60)
    print("예측 기간 설정")
    print("="*60)
    print(f"📅 예측 시작: {PREDICTION_START_YEAR}년 {PREDICTION_START_MONTH}월 (고정)")
    print(f"   현재 데이터가 {LAST_DATA_YEAR}년 {LAST_DATA_MONTH}월까지 있어서 {PREDICTION_START_MONTH}월부터 예측 가능합니다.")
    print()
    print("어디까지 예측하고 싶으신지 알려주세요:")
    print()
    print("💡 권장 예측 기간:")
    print("   - 단기 예측: 6개월~1년 (높은 정확도)")
    print("   - 중기 예측: 1~3년 (보통 정확도)")  
    print("   - 장기 예측: 3~5년 (참고용)")
    print()
    
    # 마지막 예측 년도 선택
    while True:
        try:
            end_year = int(input(f"마지막 예측 년도 ({PREDICTION_START_YEAR}-{MAX_PREDICTION_YEAR}): "))
            if end_year < PREDICTION_START_YEAR:
                print(f"{end_year}년은 예측 시작년도({PREDICTION_START_YEAR})보다 이전입니다.")
            elif end_year > MAX_PREDICTION_YEAR:
                print(f"{end_year}년은 예측 범위를 초과합니다. {MAX_PREDICTION_YEAR}년까지 가능합니다.")
            else:
                selected_end_year = end_year
                print(f"선택됨: {end_year}년")
                break
        except ValueError:
            print("올바른 숫자를 입력하세요.")
    
    # 마지막 예측 월 선택
    if selected_end_year == PREDICTION_START_YEAR:
        # 같은 년도인 경우 시작 월 이후만 가능
        min_month = PREDICTION_START_MONTH
        print(f"마지막 예측 월 ({min_month}-12): ", end="")
    else:
        # 다른 년도인 경우 모든 월 가능
        min_month = 1
        print(f"마지막 예측 월 (1-12): ", end="")
    
    while True:
        try:
            end_month = int(input())
            if selected_end_year == PREDICTION_START_YEAR and end_month < PREDICTION_START_MONTH:
                print(f"{selected_end_year}년 {end_month}월은 예측 시작({PREDICTION_START_MONTH}월)보다 이전입니다.")
                print(f"마지막 예측 월 ({min_month}-12): ", end="")
            elif end_month < 1 or end_month > 12:
                print("1~12 사이의 숫자를 입력하세요.")
                print(f"마지막 예측 월 ({min_month}-12): ", end="")
            else:
                selected_end_month = end_month
                print(f"선택됨: {end_month}월")
                break
        except ValueError:
            print("올바른 숫자를 입력하세요.")
            print(f"마지막 예측 월 ({min_month}-12): ", end="")
    
    # 예측 기간 계산 및 확인
    start_date = pd.to_datetime(f'{PREDICTION_START_YEAR}-{PREDICTION_START_MONTH:02d}-01')
    end_date = pd.to_datetime(f'{selected_end_year}-{selected_end_month:02d}-01')
    
    # 월 차이 계산
    total_months = (selected_end_year - PREDICTION_START_YEAR) * 12 + (selected_end_month - PREDICTION_START_MONTH) + 1
    years = total_months // 12
    months = total_months % 12
    
    if years > 0 and months > 0:
        period_str = f"{years}년 {months}개월"
    elif years > 0:
        period_str = f"{years}년"
    else:
        period_str = f"{months}개월"
    
    print(f"\n예측 구간 확정: {PREDICTION_START_YEAR}년 {PREDICTION_START_MONTH}월 ~ {selected_end_year}년 {selected_end_month}월")
    print(f"총 예측 기간: {total_months}개월 ({period_str})")
    
    # 결과 요약
    print("\n" + "="*60)
    print("분석 조건 확정")
    print("="*60)
    print(f"정상화 방법: {normalization} ({normalization_methods[normalization]})")
    print(f"국적: {selected_nationality}")
    print(f"목적: {selected_purpose}")
    print(f"계절: {selected_season}")
    print(f"예측 구간: {PREDICTION_START_YEAR}년 {PREDICTION_START_MONTH}월 ~ {selected_end_year}년 {selected_end_month}월")
    print(f"예측 기간: {total_months}개월 ({period_str})")
    print("="*60)
    
    return {
        'normalization': normalization,
        'nationality': selected_nationality,
        'purpose': selected_purpose,
        'season': selected_season,
        'start_year': PREDICTION_START_YEAR,
        'start_month': PREDICTION_START_MONTH,
        'end_year': selected_end_year,
        'end_month': selected_end_month,
        'total_months': total_months
    }

# 사용자 입력 받기
user_input = get_user_input()

외국인 입국자 예측 분석
현재 데이터: 2025년 5월까지

정상화 방법:
  0: 코로나 기간 제외
  1: 트렌드 보정
  2: 구조적 변화 모델링
  3: 외삽법
  4: 코로나 포함 전체
  5: 브리지 모델링
정상화 방법 선택 (0-5): 5
선택됨: 브리지 모델링

사용 가능한 국적: 61개
예시: 그리스, 나이지리아, 네덜란드, 노르웨이, 뉴질랜드 ...
국적 선택 (엔터=전체): 
선택됨: 전체 국적

사용 가능한 목적: 공용, 관광, 상용, 유학연수
목적 선택 (엔터=전체): 
선택됨: 전체 목적

사용 가능한 계절: 봄, 여름, 가을, 겨울
계절 선택 (엔터=전체): 
선택됨: 전체 계절

예측 기간 설정
📅 예측 시작: 2025년 6월 (고정)
   현재 데이터가 2025년 5월까지 있어서 6월부터 예측 가능합니다.

어디까지 예측하고 싶으신지 알려주세요:

💡 권장 예측 기간:
   - 단기 예측: 6개월~1년 (높은 정확도)
   - 중기 예측: 1~3년 (보통 정확도)
   - 장기 예측: 3~5년 (참고용)

마지막 예측 년도 (2025-2030): 2026
선택됨: 2026년
마지막 예측 월 (1-12): 12
선택됨: 12월

예측 구간 확정: 2025년 6월 ~ 2026년 12월
총 예측 기간: 19개월 (1년 7개월)

분석 조건 확정
정상화 방법: 5 (브리지 모델링)
국적: 전체
목적: 전체
계절: 전체
예측 구간: 2025년 6월 ~ 2026년 12월
예측 기간: 19개월 (1년 7개월)


In [5]:
def calculate_metrics(y_true, y_pred, model_name=""):
    """모델 성능 지표 계산"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    # R² 계산 (분모가 0인 경우 처리)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
    
    # MAPE 계산 (0으로 나누기 방지)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else float('inf')
    
    return {
        'model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2,
        'MAPE': mape
    }

def filter_data_by_conditions(df, conditions):
    """사용자 선택 조건에 따라 데이터 필터링 (수정버전)"""
    filtered_df = df.copy()
    
    # 국적 필터링
    if conditions['nationality'] != "전체":
        filtered_df = filtered_df[filtered_df['국적'] == conditions['nationality']]
    
    # 목적 필터링
    if conditions['purpose'] != "전체":
        filtered_df = filtered_df[filtered_df['목적'] == conditions['purpose']]
    
    # 계절 필터링
    if conditions['season'] != "전체":
        filtered_df = filtered_df[filtered_df['계절'] == conditions['season']]
    
    # 예측 구간 설정 (시작년월 ~ 끝년월)
    start_year = conditions['start_year']
    start_month = conditions['start_month'] 
    end_year = conditions['end_year']
    end_month = conditions['end_month']
    
    # 예측 대상: 시작년월 ~ 끝년월 사이의 모든 월
    prediction_mask = (
        ((filtered_df['연도'] == start_year) & (filtered_df['월'] >= start_month)) |
        ((filtered_df['연도'] > start_year) & (filtered_df['연도'] < end_year)) |
        ((filtered_df['연도'] == end_year) & (filtered_df['월'] <= end_month))
    )
    
    # 훈련 데이터: 예측 시작 이전의 모든 데이터
    training_mask = (
        (filtered_df['연도'] < start_year) |
        ((filtered_df['연도'] == start_year) & (filtered_df['월'] < start_month))
    )
    
    train_data = filtered_df[training_mask]
    prediction_target = filtered_df[prediction_mask]
    
    print(f"필터링 결과:")
    print(f"  전체 데이터: {len(filtered_df):,}개")
    print(f"  훈련 데이터: {len(train_data):,}개")
    print(f"  예측 대상: {len(prediction_target):,}개 ({start_year}년 {start_month}월 ~ {end_year}년 {end_month}월)")
    
    return train_data, prediction_target

def create_train_test_split(df, normalization_method=0):
    """시계열 데이터 분할 (정상화 방법에 따라 조정)"""
    df_sorted = df.sort_values(['국적', '목적', 'date'])
    
    if normalization_method == 0:
        # 방법 0: 코로나 기간을 훈련 데이터에서 제외
        print("훈련 데이터에서 코로나 기간(2020-2023) 제외")
        train_data = []
        test_data = []
        
        for (nationality, purpose), group in df_sorted.groupby(['국적', '목적']):
            # 정상 기간 데이터만 훈련용으로 사용
            normal_group = group[group['코로나기간'] == 0]
            
            if len(normal_group) > 12:  # 최소 1년 데이터 필요
                # 정상 기간에서 train/test 분할 (80:20)
                split_idx = int(len(normal_group) * 0.8)
                train_group = normal_group.iloc[:split_idx]
                test_group = normal_group.iloc[split_idx:]
                
                if len(train_group) > 0:
                    train_data.append(train_group)
                if len(test_group) > 0:
                    test_data.append(test_group)
    
    else:
        # 방법 1,2,3,4,5: 전체 데이터 사용하되 정상화 적용된 상태
        print(f"정상화 방법 {normalization_method} 적용된 전체 데이터 사용")
        train_data = []
        test_data = []
        
        for (nationality, purpose), group in df_sorted.groupby(['국적', '목적']):
            if len(group) > 12:
                # 전체 데이터에서 train/test 분할 (80:20)
                split_idx = int(len(group) * 0.8)
                train_group = group.iloc[:split_idx]
                test_group = group.iloc[split_idx:]
                train_data.append(train_group)
                test_data.append(test_group)
    
    train_df = pd.concat(train_data, ignore_index=True) if train_data else pd.DataFrame()
    test_df = pd.concat(test_data, ignore_index=True) if test_data else pd.DataFrame()
    
    return train_df, test_df

def create_features(df, target_col='입국자수'):
    """시계열 특징 생성"""
    features = df.copy()
    
    # 기본 특징
    feature_cols = [
        '국적_encoded', '목적_encoded', '연도', '월', '분기',
        '계절_sin', '계절_cos', '월_sin', '월_cos', 
        '코로나기간', '시계열순서'
    ]
    
    # 정상화 방법에 따른 추가 특징
    if 'normalization_method' in features.columns:
        norm_method = features['normalization_method'].iloc[0] if len(features) > 0 else 0
        
        if norm_method == 2:  # 구조적 변화 모델링
            if 'regime' in features.columns:
                # regime을 더미 변수로 변환
                features['regime_covid'] = (features['regime'] == 'covid').astype(int)
                feature_cols.append('regime_covid')
            
            if 'recovery_factor' in features.columns:
                feature_cols.append('recovery_factor')
    
    # 기존 lag features 사용 (코로나 기간 제외 시 일부 누락 가능)
    if '입국자수_1개월전' in df.columns:
        feature_cols.extend(['입국자수_1개월전', '입국자수_3개월전', '입국자수_12개월전'])
    
    # 기존 이동평균 사용
    if '입국자수_3개월평균' in df.columns:
        feature_cols.extend(['입국자수_3개월평균', '입국자수_12개월평균'])
    
    # 기존 증감률 사용
    if '전년동월대비증감률' in df.columns:
        feature_cols.append('전년동월대비증감률')
    
    # 실제 존재하는 컬럼만 선택
    available_cols = [col for col in feature_cols if col in features.columns]
    
    # 결측치 처리
    for col in available_cols:
        if col in features.columns:
            features[col] = features[col].fillna(0)
    
    return features[available_cols], features[target_col]

print("전처리 함수 정의 완료")

전처리 함수 정의 완료


In [6]:
def train_prophet_model(train_df, prediction_target_df):
    """Prophet 모델 훈련 및 예측 (수정버전)"""
    if not prophet_available:
        print("Prophet을 사용할 수 없습니다.")
        return None, None, None
    
    start_time = time.time()
    predictions = []
    
    print("Prophet 모델 훈련 중...")
    
    # 각 국적-목적 조합별로 모델 훈련
    group_count = 0
    for (nationality, purpose), train_group in train_df.groupby(['국적', '목적']):
        group_count += 1
        
        try:
            # 해당 조합의 예측 대상 확인
            prediction_group = prediction_target_df[
                (prediction_target_df['국적'] == nationality) & 
                (prediction_target_df['목적'] == purpose)
            ]
            
            if len(prediction_group) == 0:
                continue
                
            # Prophet 데이터 형식으로 변환
            prophet_df = train_group[['date', '입국자수']].rename(columns={'date': 'ds', '입국자수': 'y'})
            prophet_df = prophet_df.sort_values('ds')
            
            # 최소 데이터 요구사항 확인
            if len(prophet_df) < 12:
                continue
                
            # 모델 생성 및 훈련
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=0.1,
                seasonality_prior_scale=10.0,
                interval_width=0.8
            )
            
            model.fit(prophet_df)
            
            # 예측할 날짜 생성
            future_dates = prediction_group[['date']].rename(columns={'date': 'ds'})
            future_dates = future_dates.sort_values('ds')
            forecast = model.predict(future_dates)
            
            # 예측 결과 저장
            for i, (_, row) in enumerate(prediction_group.iterrows()):
                pred_idx = future_dates[future_dates['ds'] == row['date']].index
                if len(pred_idx) > 0:
                    predicted_value = forecast.loc[forecast.index[i], 'yhat']
                    predictions.append({
                        'actual': row['입국자수'],  # 실제로는 예측할 값 (미래)
                        'predicted': max(0, predicted_value),  # 음수 방지
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date']
                    })
                    
        except Exception as e:
            if group_count <= 5:  # 처음 5개 에러만 출력
                print(f"Prophet 에러 (그룹 {group_count}): {str(e)}")
            continue
    
    training_time = time.time() - start_time
    
    if predictions:
        pred_df = pd.DataFrame(predictions)
        # 예측 성능은 실제 미래값이 없으므로 더미 메트릭 생성
        dummy_metrics = {
            'model': 'Prophet',
            'MAE': len(predictions),  # 예측 개수로 대체
            'MSE': training_time,     # 훈련 시간으로 대체  
            'RMSE': training_time,
            'R²': min(1.0, len(predictions) / 100),  # 더미 R²
            'MAPE': max(1.0, 100 / len(predictions))  # 더미 MAPE
        }
        print(f"Prophet 완료 - 예측 수: {len(predictions)}개, 시간: {training_time:.2f}초")
        return dummy_metrics, pred_df, training_time
    else:
        print("Prophet 예측 결과가 없습니다.")
        return None, None, training_time


# Prophet 모델 실행 
print("Prophet 모델 테스트 실행...")

Prophet 모델 테스트 실행...


In [7]:
def train_sarima_model(train_df, test_df, max_groups=10, use_auto_arima=True):
    """SARIMA 모델 훈련 및 예측 (수정버전)"""
    if not statsmodels_available:
        print("Statsmodels를 사용할 수 없습니다.")
        return None, None, None
    
    start_time = time.time()
    predictions = []
    
    print("SARIMA 모델 훈련 중...")
    
    # 계산 시간 단축을 위한 샘플링
    groups = list(train_df.groupby(['국적', '목적']))
    if len(groups) > max_groups:
        import random
        random.seed(42)
        groups = random.sample(groups, max_groups)
        print(f"계산 시간 단축을 위해 {max_groups}개 그룹만 샘플링")
    
    group_count = 0
    for (nationality, purpose), group in groups:
        group_count += 1
        
        try:
            # 해당 조합의 테스트 데이터 확인
            test_group = test_df[(test_df['국적'] == nationality) & (test_df['목적'] == purpose)]
            
            if len(group) < 24 or len(test_group) == 0:  # 최소 2년 데이터 필요
                continue
            
            # 시계열 데이터 준비
            ts_data = group.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            ts_data = ts_data.astype(float)
            
            # SARIMA 모델 선택
            if use_auto_arima:
                # auto_arima 사용 (더 정확하지만 느림)
                try:
                    from pmdarima import auto_arima
                    auto_model = auto_arima(ts_data, 
                                          seasonal=True, 
                                          m=12,
                                          max_p=3, max_q=3, max_P=2, max_Q=2,
                                          max_d=2, max_D=1,
                                          stepwise=True,
                                          suppress_warnings=True,
                                          error_action='ignore',
                                          trace=False)
                    fitted_model = auto_model
                except ImportError:
                    print("pmdarima가 설치되지 않음. 기본 SARIMA 사용")
                    use_auto_arima = False
            
            if not use_auto_arima:
                # 기본 SARIMA 모델 (빠르지만 덜 정확)
                model = SARIMAX(ts_data, 
                              order=(1, 1, 1), 
                              seasonal_order=(1, 1, 1, 12),
                              enforce_stationarity=False,
                              enforce_invertibility=False)
                fitted_model = model.fit(disp=False, maxiter=50)
            
            # 예측
            forecast = fitted_model.forecast(steps=len(test_group))
            
            # 예측 결과 저장
            for i, (_, row) in enumerate(test_group.iterrows()):
                if i < len(forecast):
                    predictions.append({
                        'actual': row['입국자수'],
                        'predicted': max(0, forecast.iloc[i] if hasattr(forecast, 'iloc') else forecast[i]),
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date']
                    })
                    
        except Exception as e:
            if group_count <= 3:  # 처음 3개 에러만 출력
                print(f"SARIMA 에러 (그룹 {group_count}): {str(e)}")
            continue
    
    training_time = time.time() - start_time
    
    if predictions:
        pred_df = pd.DataFrame(predictions)
        y_true = pred_df['actual'].values
        y_pred = pred_df['predicted'].values
        metrics = calculate_metrics(y_true, y_pred, 'SARIMA')
        print(f"SARIMA 완료 - MAPE: {metrics['MAPE']:.2f}%, 시간: {training_time:.2f}초")
        return metrics, pred_df, training_time
    else:
        print("SARIMA 예측 결과가 없습니다.")
        return None, None, training_time

# 모델 실행 예시
print("SARIMA 모델 테스트 실행 (기본: auto_arima 최적화 모드)")

SARIMA 모델 테스트 실행 (기본: auto_arima 최적화 모드)


In [8]:
def train_ets_model(train_df, test_df, max_groups=10):
    """ETS 모델 훈련 및 예측"""
    if not statsmodels_available:
        print("Statsmodels를 사용할 수 없습니다.")
        return None, None, None
    
    start_time = time.time()
    predictions = []
    
    print("ETS 모델 훈련 중...")
    
    # 계산 시간 단축을 위한 샘플링
    groups = list(train_df.groupby(['국적', '목적']))
    if len(groups) > max_groups:
        import random
        random.seed(42)
        groups = random.sample(groups, max_groups)
        print(f"계산 시간 단축을 위해 {max_groups}개 그룹만 샘플링")
    
    group_count = 0
    for (nationality, purpose), group in groups:
        group_count += 1
        
        try:
            test_group = test_df[(test_df['국적'] == nationality) & (test_df['목적'] == purpose)]
            
            if len(group) < 24 or len(test_group) == 0:
                continue
            
            # 시계열 데이터 준비
            ts_data = group.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            ts_data = ts_data.astype(float)
            
            # ETS 모델
            model = ETSModel(ts_data, 
                           error='add', 
                           trend='add', 
                           seasonal='add', 
                           seasonal_periods=12)
            
            fitted_model = model.fit(maxiter=100)
            
            # 예측
            forecast = fitted_model.forecast(steps=len(test_group))
            
            for i, (_, row) in enumerate(test_group.iterrows()):
                if i < len(forecast):
                    predictions.append({
                        'actual': row['입국자수'],
                        'predicted': max(0, forecast.iloc[i] if hasattr(forecast, 'iloc') else forecast[i]),
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date']
                    })
                    
        except Exception as e:
            print(f"ETS 에러 (그룹 {group_count}): {str(e)}")
            continue
    
    training_time = time.time() - start_time
    
    if predictions:
        pred_df = pd.DataFrame(predictions)
        y_true = pred_df['actual'].values
        y_pred = pred_df['predicted'].values
        metrics = calculate_metrics(y_true, y_pred, 'ETS')
        print(f"ETS 완료 - MAPE: {metrics['MAPE']:.2f}%, 시간: {training_time:.2f}초")
        return metrics, pred_df, training_time
    else:
        print("ETS 예측 결과가 없습니다.")
        return None, None, training_time

# ETS 모델 실행 (예시)
print("ETS 모델 테스트 실행...")

ETS 모델 테스트 실행...


In [10]:
# Cell 9 (수정: 기존 9번 셀의 내용을 이 코드로 교체하세요)

# --- 0. 라이브러리 가용성 체크 (최신 버전으로 업데이트) ---
# 이 부분은 필요시 2-5번 셀의 기존 임포트 및 체크 부분과 병합하거나 대체할 수 있습니다.
# 여기서는 필요한 라이브러리가 미리 임포트되어 있다고 가정합니다.
# import pandas as pd
# import numpy as np
# import time
# from sklearn.metrics import mean_absolute_percentage_error # MAPE 계산용 (없으면 calculate_metrics 함수 안에 직접 구현)

prophet_available = False
try:
    from prophet import Prophet
    prophet_available = True
except ImportError:
    print("경고: Prophet 라이브러리를 찾을 수 없습니다. 설치가 필요할 수 있습니다 (pip install prophet).")

statsmodels_available = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.api import ETSModel
    statsmodels_available = True
except ImportError:
    print("경고: Statsmodels 라이브러리를 찾을 수 없습니다. 설치가 필요할 수 있습니다 (pip install statsmodels).")

pmdarima_available = False
try:
    from pmdarima import auto_arima
    pmdarima_available = True
except ImportError:
    print("경고: pmdarima 라이브러리를 찾을 수 없습니다. auto_arima를 사용할 수 없습니다 (pip install pmdarima).")

# --- 3. calculate_metrics 함수 정의 (업데이트된 MAPE 계산 포함) ---
# **주의: 이 함수는 이전 셀의 calculate_metrics를 대체합니다.**
def calculate_metrics(y_true, y_pred):
    # NaN 값 제거 (예측값과 실제값 중 하나라도 NaN이면 제외)
    valid_indices = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true_clean = y_true[valid_indices]
    y_pred_clean = y_pred[valid_indices]

    if len(y_true_clean) == 0:
        return None # 유효한 값이 없으면 metrics 계산 불가

    # MAPE 계산 (0으로 나누는 오류 방지)
    small_epsilon = 1e-8 # 0으로 나누는 것을 방지하기 위한 작은 값
    mape = np.mean(np.abs((y_true_clean - y_pred_clean) / (y_true_clean + small_epsilon))) * 100

    return {'MAPE': mape}

# --- 4. `apply_normalization`, `filter_data_by_conditions` 함수 정의 (업데이트된 내용) ---
# **주의: 이 함수들은 이전 셀의 동일 이름 함수를 대체합니다.**
def apply_normalization(df, norm_method):
    print(f"정상화 방법 {norm_method} 적용 중...")
    df_copy = df.copy() # 원본 데이터프레임 복사
    
    if norm_method == 0: # 코로나 기간 (2020년 ~ 2022년) 제외
        df_copy = df_copy[(df_copy['date'] < '2020-01-01') | (df_copy['date'] > '2022-12-31')].copy()
    elif norm_method == 1: # 트렌드 보정 (예시: 로그 변환)
        # 1을 더한 후 로그 변환 (0 또는 음수 방지)
        df_copy['입국자수'] = np.log1p(df_copy['입국자수'].replace([np.inf, -np.inf], np.nan).fillna(0)) 
    elif norm_method == 2: # 구조적 변화 모델링 (예시: 2018년 이후 데이터만 사용)
        df_copy = df_copy[df_copy['date'] >= '2018-01-01'].copy()
    elif norm_method == 3: # 외삽법 (가장 최근 데이터에 가중치)
        # 이 부분은 실제 모델 학습 시 가중치를 적용하는 방식으로 이루어져야 함.
        # 여기서는 단순히 데이터를 그대로 반환하며, 모델이 학습 시 외삽 특성을 학습하도록 함.
        pass
    elif norm_method == 4: # 코로나 포함 전체
        pass # 전체 데이터 사용
    elif norm_method == 5: # 브리지 모델링 (예시: 계절성만 강조)
        df_copy['입국자수'] = df_copy.groupby(df_copy['date'].dt.month)['입국자수'].transform(lambda x: x / x.mean())
    
    return df_copy

def filter_data_by_conditions(df, user_input_params): # user_input과의 이름 충돌 방지를 위해 user_input_params로 변경
    # 훈련 데이터: 전체 데이터 중 2024년 12월까지
    train_data = df[df['date'] < '2025-01-01'].copy()

    # 검증 데이터 (prediction_target): 2025년 1월부터 2025년 5월까지의 실제 값이 존재하는 구간
    # 이 구간의 실제 '입국자수' 값을 기준으로 모델의 성능(MAPE 등)을 평가
    prediction_target_validation = df[(df['date'] >= '2025-01-01') & (df['date'] <= '2025-05-31')].copy()

    print(f"필터링 결과:")
    print(f"  전체 데이터: {len(df)}개")
    print(f"  훈련 데이터: {len(train_data)}개 ({train_data['date'].min().strftime('%Y-%m')} ~ {train_data['date'].max().strftime('%Y-%m') if not train_data.empty else 'N/A'})")
    if not prediction_target_validation.empty:
        print(f"  예측 대상 (검증): {len(prediction_target_validation)}개 ({prediction_target_validation['date'].min().strftime('%Y-%m')} ~ {prediction_target_validation['date'].max().strftime('%Y-%m')})")
    else:
        print(f"  예측 대상 (검증): 0개 (이 기간에 실제 값이 없으면 메트릭스 계산 불가)")

    return train_data, prediction_target_validation


# --- 5. `train_prophet_model` 함수 정의 ---
def train_prophet_model(train_df, prediction_target_df):
    """Prophet 모델 훈련 및 예측"""
    if not prophet_available:
        print("Prophet을 사용할 수 없습니다.")
        return None, None, None # metrics, predictions, training_time

    start_time = time.time()
    predictions_list = [] # 모든 그룹의 예측 결과를 모을 리스트

    print("Prophet 모델 훈련 중...")

    # 각 국적-목적 조합별로 모델 훈련
    group_count = 0
    # train_df와 prediction_target_df를 date 기준으로 정렬 보장
    train_df = train_df.sort_values('date')
    prediction_target_df = prediction_target_df.sort_values('date')

    for (nationality, purpose), train_group in train_df.groupby(['국적', '목적']):
        group_count += 1
        
        # 해당 그룹의 훈련 데이터와 예측 대상 확인
        prediction_group = prediction_target_df[
            (prediction_target_df['국적'] == nationality) &
            (prediction_target_df['목적'] == purpose)
        ]
        
        # 이 그룹에 대한 예측 대상이 없으면 스킵
        if len(prediction_group) == 0:
            continue
            
        # Prophet 데이터 형식으로 변환
        prophet_df = train_group[['date', '입국자수']].rename(columns={'date': 'ds', '입국자수': 'y'})
        prophet_df = prophet_df.sort_values('ds')
        
        # 최소 데이터 요구사항 확인 (Prophet은 최소 2개 이상, 계절성 위해선 더 많은 데이터 필요)
        if len(prophet_df) < 24: # 최소 2년치 월별 데이터 (계절성 충분히 파악)
            # print(f"경고: {nationality}-{purpose} 그룹은 데이터가 너무 적어 (현재 {len(prophet_df)}개) Prophet 학습을 건너뜜.")
            continue
            
        try:
            # 모델 생성 및 훈련
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False, # 월별 데이터이므로 Daily는 의미 없음
                daily_seasonality=False, # 월별 데이터이므로 Weekly는 의미 없음
                changepoint_prior_scale=0.1,
                seasonality_prior_scale=10.0,
                interval_width=0.8
            )
            model.fit(prophet_df)
            
            # 예측할 날짜 생성
            future_dates_df = prediction_group[['date']].rename(columns={'date': 'ds'})
            future_dates_df = future_dates_df.sort_values('ds')
            
            # 예측 수행
            forecast = model.predict(future_dates_df)
            
            # 예측 결과 저장
            # prediction_group과 forecast를 ds/date 컬럼을 기준으로 병합하여 정확한 실제값-예측값 매칭
            merged_pred = pd.merge(prediction_group, forecast[['ds', 'yhat']],
                                   left_on='date', right_on='ds', how='left')
            
            for _, row in merged_pred.iterrows():
                if pd.notna(row['yhat']): # 예측값이 유효한 경우에만 추가
        
                    # 적용된 정상화 방법 확인 (이 그룹의 normalization_method 컬럼을 활용)
                    # train_group은 이 함수 내에서 이미 정의되어 있습니다.
                    # prediction_target_df에도 normalization_method 컬럼이 있을 것으로 가정합니다.
                    # 여기서는 train_group의 normalization_method를 사용합니다.
                    current_norm_method = train_group['normalization_method'].iloc[0] if 'normalization_method' in train_group.columns else -1
        
                    predicted_value = row['yhat']
        
                    # norm_method 1 (로그 변환)인 경우 역변환
                    if current_norm_method == 1:
                        predicted_value = np.expm1(predicted_value) # np.expm1은 np.log1p의 역함수
        
                    predictions_list.append({
                        'actual': row['입국자수'],
                        'predicted': max(0, predicted_value), # 음수 방지 및 역변환 적용
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date']
                    })
                    
        except Exception as e:
            if group_count <= 5: # 처음 5개 에러만 출력하여 로그 과도화 방지
                print(f"Prophet 에러 ({nationality}-{purpose} 그룹): {str(e)}")
            continue

    training_time = time.time() - start_time

    if predictions_list:
        pred_df = pd.DataFrame(predictions_list)
        
        # 실제 값이 존재하는 경우에만 metrics 계산
        y_true = pred_df['actual'].dropna().values
        y_pred = pred_df.loc[pred_df['actual'].dropna().index, 'predicted'].values

        metrics_calculated = None
        if len(y_true) > 0:
            metrics_calculated = calculate_metrics(y_true, y_pred)
            print(f"Prophet 완료 - 예측 수: {len(predictions_list)}개, 시간: {training_time:.2f}초, MAPE: {metrics_calculated.get('MAPE', 'N/A'):.2f}%")
        else:
            print(f"Prophet 완료 - 예측 수: {len(predictions_list)}개, 시간: {training_time:.2f}초 (메트릭스 계산 불가 - 실제 값 없음)")

        return metrics_calculated, pred_df, training_time
    else:
        print("Prophet 예측 결과가 없습니다.")
        return None, None, training_time

# --- SARIMA 모델 훈련 함수 ---
def train_sarima_model(train_df, prediction_target_df, max_groups=10, use_auto_arima=True):
    """SARIMA 모델 훈련 및 예측"""
    if not statsmodels_available:
        print("Statsmodels를 사용할 수 없습니다.")
        return None, None, None # metrics, predictions, training_time
    
    # auto_arima 사용 여부 재확인
    if use_auto_arima and not pmdarima_available:
        print("pmdarima 라이브러리를 찾을 수 없습니다. auto_arima를 사용할 수 없습니다. 기본 SARIMA 사용.")
        use_auto_arima = False

    start_time = time.time()
    predictions_list = []

    print("SARIMA 모델 훈련 중...")

    groups = list(train_df.groupby(['국적', '목적']))
    if len(groups) > max_groups: # 계산 시간 단축을 위한 샘플링
        import random
        random.seed(42) # 재현성을 위해 시드 설정
        groups = random.sample(groups, max_groups)
        print(f"계산 시간 단축을 위해 {max_groups}개 그룹만 샘플링")
    
    group_count = 0
    from statsmodels.tsa.statespace.sarimax import SARIMAX # 함수 내에서 임포트

    for (nationality, purpose), train_group in groups:
        group_count += 1
        try:
            prediction_group = prediction_target_df[
                (prediction_target_df['국적'] == nationality) &
                (prediction_target_df['목적'] == purpose)
            ]
            
            if len(train_group) < 24 or len(prediction_group) == 0: # 최소 2년 데이터 필요
                continue
            
            # 시계열 데이터 준비 (날짜를 인덱스로 설정)
            ts_data = train_group.set_index('date')['입국자수'].asfreq('MS', fill_value=0) # 월 시작 빈도
            ts_data = ts_data.astype(float)
            
            fitted_model = None
            if use_auto_arima:
                try:
                    fitted_model = auto_arima(ts_data, 
                                              seasonal=True, m=12,
                                              max_p=3, max_q=3, max_P=2, max_Q=2,
                                              max_d=2, max_D=1,
                                              stepwise=True,
                                              suppress_warnings=True,
                                              error_action='ignore',
                                              trace=False)
                except Exception as auto_e:
                    print(f"Auto-ARIMA 에러 ({nationality}-{purpose}): {str(auto_e)}. 기본 SARIMA 시도.")
                    use_auto_arima = False # auto_arima 실패 시 기본 SARIMA로 대체

            if not use_auto_arima or fitted_model is None: # auto_arima를 사용하지 않거나 실패한 경우 기본 SARIMA
                model = SARIMAX(ts_data, 
                                order=(1, 1, 1), 
                                seasonal_order=(1, 1, 1, 12),
                                enforce_stationarity=False,
                                enforce_invertibility=False)
                fitted_model = model.fit(disp=False, maxiter=50)
            
            # 예측 수행 (prediction_group의 날짜 범위에 맞춰)
            forecast_start_date = ts_data.index.max() + pd.DateOffset(months=1)
            forecast_end_date = prediction_group['date'].max()

            # predict() 메소드의 start, end 인자는 시계열 인덱스의 위치가 아닌, 날짜/시간 값이어야 함
            forecast_series = fitted_model.predict(start=forecast_start_date, 
                                                   end=forecast_end_date,
                                                   typ='levels')
            
            # 예측 결과 저장 (날짜를 기준으로 prediction_group의 실제 값과 매칭)
            merged_pred = pd.merge(prediction_group, forecast_series.rename('yhat'),
                                   left_on='date', right_index=True, how='left')

            for _, row in merged_pred.iterrows():
                if pd.notna(row['yhat']):
        
                    # 적용된 정상화 방법 확인 (train_group의 normalization_method 컬럼을 활용)
                    current_norm_method = train_group['normalization_method'].iloc[0] if 'normalization_method' in train_group.columns else -1
        
                    predicted_value = row['yhat']
        
                    # norm_method 1 (로그 변환)인 경우 역변환
                    if current_norm_method == 1:
                        predicted_value = np.expm1(predicted_value)
        
                    predictions_list.append({
                        'actual': row['입국자수'],
                        'predicted': max(0, predicted_value), # 음수 방지 및 역변환 적용
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date']
                    })

        except Exception as e:
            if group_count <= 3: # 처음 3개 에러만 출력
                print(f"SARIMA 에러 ({nationality}-{purpose} 그룹): {str(e)}")
            continue

    training_time = time.time() - start_time

    if predictions_list:
        pred_df = pd.DataFrame(predictions_list)
        y_true = pred_df['actual'].dropna().values
        y_pred = pred_df.loc[pred_df['actual'].dropna().index, 'predicted'].values
        
        metrics_calculated = None
        if len(y_true) > 0:
            metrics_calculated = calculate_metrics(y_true, y_pred)
            print(f"SARIMA 완료 - 예측 수: {len(predictions_list)}개, 시간: {training_time:.2f}초, MAPE: {metrics_calculated.get('MAPE', 'N/A'):.2f}%")
        else:
            print(f"SARIMA 완료 - 예측 수: {len(predictions_list)}개, 시간: {training_time:.2f}초 (메트릭스 계산 불가 - 실제 값 없음)")

        return metrics_calculated, pred_df, training_time
    else:
        print("SARIMA 예측 결과가 없습니다.")
        return None, None, training_time


# --- ETS 모델 훈련 함수 ---
def train_ets_model(train_df, prediction_target_df, max_groups=10):
    """ETS 모델 훈련 및 예측"""
    if not statsmodels_available:
        print("Statsmodels를 사용할 수 없습니다.")
        return None, None, None # metrics, predictions, training_time

    start_time = time.time()
    predictions_list = []

    print("ETS 모델 훈련 중...")

    groups = list(train_df.groupby(['국적', '목적']))
    if len(groups) > max_groups:
        import random
        random.seed(42)
        groups = random.sample(groups, max_groups)
        print(f"계산 시간 단축을 위해 {max_groups}개 그룹만 샘플링")

    group_count = 0
    from statsmodels.tsa.api import ETSModel # 함수 내에서 임포트
    
    for (nationality, purpose), train_group in groups:
        group_count += 1
        try:
            prediction_group = prediction_target_df[
                (prediction_target_df['국적'] == nationality) &
                (prediction_target_df['목적'] == purpose)
            ]
            
            if len(train_group) < 24 or len(prediction_group) == 0:
                continue
            
            # 시계열 데이터 준비 (날짜를 인덱스로 설정)
            ts_data = train_group.set_index('date')['입국자수'].asfreq('MS', fill_value=0)
            ts_data = ts_data.astype(float)
            
            model = ETSModel(ts_data, 
                             error='add', 
                             trend='add', 
                             seasonal='add', 
                             seasonal_periods=12) # 월별 데이터이므로 12
            
            fitted_model = model.fit(maxiter=100)
            
            # 예측 수행 (prediction_group의 날짜 범위에 맞춰)
            forecast_steps = len(prediction_group)
            forecast_series = fitted_model.forecast(steps=forecast_steps)
            
            # forecast_series의 인덱스가 prediction_group의 date와 일치하도록 설정
            forecast_series.index = prediction_group['date'].sort_values()

            # 예측 결과 저장 (날짜를 기준으로 prediction_group의 실제 값과 매칭)
            merged_pred = pd.merge(prediction_group, forecast_series.rename('yhat'),
                                   left_on='date', right_index=True, how='left')

            for _, row in merged_pred.iterrows():
                if pd.notna(row['yhat']):
        
                    # 적용된 정상화 방법 확인 (train_group의 normalization_method 컬럼을 활용)
                    current_norm_method = train_group['normalization_method'].iloc[0] if 'normalization_method' in train_group.columns else -1
        
                    predicted_value = row['yhat']
        
                    # norm_method 1 (로그 변환)인 경우 역변환
                    if current_norm_method == 1:
                        predicted_value = np.expm1(predicted_value)
        
                    predictions_list.append({
                        'actual': row['입국자수'],
                        'predicted': max(0, predicted_value), # 음수 방지 및 역변환 적용
                        'nationality': nationality,
                        'purpose': purpose,
                        'date': row['date']
                    })

        except Exception as e:
            print(f"ETS 에러 ({nationality}-{purpose} 그룹): {str(e)}")
            continue

    training_time = time.time() - start_time

    if predictions_list:
        pred_df = pd.DataFrame(predictions_list)
        y_true = pred_df['actual'].dropna().values
        y_pred = pred_df.loc[pred_df['actual'].dropna().index, 'predicted'].values
        
        metrics_calculated = None
        if len(y_true) > 0:
            metrics_calculated = calculate_metrics(y_true, y_pred)
            print(f"ETS 완료 - 예측 수: {len(predictions_list)}개, 시간: {training_time:.2f}초, MAPE: {metrics_calculated.get('MAPE', 'N/A'):.2f}%")
        else:
            print(f"ETS 완료 - 예측 수: {len(predictions_list)}개, 시간: {training_time:.2f}초 (메트릭스 계산 불가 - 실제 값 없음)")

        return metrics_calculated, pred_df, training_time
    else:
        print("ETS 예측 결과가 없습니다.")
        return None, None, training_time

In [ ]:
# Cell 10 (새로운 셀을 만들고 이 코드를 붙여넣으세요)

# --- user_input 정의 (이전 get_user_input() 함수를 사용하지 않는다면 이 부분을 사용) ---
# 만약 이전 셀에서 get_user_input() 함수로 user_input을 받았다면, 이 부분을 주석 처리하거나 삭제하세요.
import logging

# cmdstanpy 로깅 차단
logging.getLogger("cmdstanpy").propagate = False  # 루트 로거로 전달 안함
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)  # CRITICAL 이상만 출력

# 필요하다면 Prophet 로거도 차단
logging.getLogger("prophet").setLevel(logging.CRITICAL)

user_input = {
    'prediction_start_year': 2025,
    'prediction_start_month': 6,
    'prediction_end_year': 2027,
    'prediction_end_month': 12
}
# 이 user_input은 filter_data_by_conditions 함수로 전달될 예정입니다.


# --- 6. `run_all_combinations_experiment` 함수 정의 ---
def run_all_combinations_experiment():
    """모델 학습 시작"""
    print("="*80)
    print("모델 학습 시작")
    print("="*80)

    # 결과 저장소
    all_results = {}
    model_functions = {
        'Prophet': train_prophet_model,
        'SARIMA': train_sarima_model,
        'ETS': train_ets_model
    }

    normalization_names = {
        0: "코로나 기간 제외",
        1: "로그 변환 (트렌드 보정 예시)",
        2: "2018년 이후 데이터만 (구조적 변화 예시)",
        3: "외삽법 (변환 없음)",
        4: "코로나 포함 전체",
        5: "계절성 강조 (평균 비율)"
    }

    total_combinations = len(model_functions) * len(normalization_names)
    current_combination = 0

    # 각 정상화 방법과 모델 조합 실행
    for norm_method in sorted(normalization_names.keys()):
        print(f"\n정상화 방법 {norm_method}: {normalization_names[norm_method]}")
        print("-" * 60)

        # 정상화 적용 (df_original은 이전 셀에서 로드되었다고 가정)
        # 중요: df_original은 이 셀이 실행되기 전에 올바르게 로드되어 있어야 합니다.
        df_normalized = apply_normalization(df_original.copy(), norm_method)

        # 사용자 조건에 따른 데이터 필터링 (훈련 데이터와 검증 데이터 분리)
        # user_input (전역 변수)는 이 셀이 실행되기 전에 올바르게 정의되어 있어야 합니다.
        train_data, prediction_target = filter_data_by_conditions(df_normalized, user_input)


        if len(train_data) == 0:
            print(f"정상화 방법 {norm_method}: 훈련 데이터가 없습니다. 다음 조합으로 넘어갑니다.")
            continue

        # 각 모델 훈련
        for model_name, model_func in model_functions.items():
            current_combination += 1
            print(f"\n[{current_combination}/{total_combinations}] {model_name} + 정상화{norm_method}")

            try:
                # 모델 함수 호출
                metrics, predictions_df, training_time = model_func(train_data, prediction_target)

                combination_key = f"{model_name}_norm{norm_method}"
                all_results[combination_key] = {
                    'model': model_name,
                    'normalization': norm_method,
                    'normalization_name': normalization_names[norm_method],
                    'metrics': metrics, # metrics가 None일 수 있음
                    'predictions': predictions_df, # 예측 결과 DataFrame
                    'training_time': training_time,
                    'prediction_count': len(predictions_df) if predictions_df is not None else 0
                }
                
                # 결과 출력
                if metrics is not None and 'MAPE' in metrics:
                    print(f"  완료: 예측 수 {all_results[combination_key]['prediction_count']}개, 시간 {training_time:.2f}초, MAPE: {metrics['MAPE']:.2f}%")
                else:
                    print(f"  완료 (예측만): 예측 수 {all_results[combination_key]['prediction_count']}개, 시간 {training_time:.2f}초. (메트릭스 계산 불가)")

            except Exception as e:
                print(f"  에러 발생: {model_name} + 정상화{norm_method} - {str(e)}")
                # 에러 발생 시 해당 조합은 결과에 포함하지 않거나, 실패로 기록할 수 있음
                all_results[f"{model_name}_norm{norm_method}"] = {
                    'model': model_name,
                    'normalization': norm_method,
                    'normalization_name': normalization_names[norm_method],
                    'metrics': None,
                    'predictions': None,
                    'training_time': 0, # 에러 발생 시 시간 0으로 기록
                    'prediction_count': 0
                }
                continue

    return all_results

# --- 7. `analyze_results` 함수 정의 ---
def analyze_results(all_results):
    """결과 분석 및 기법별 최고 조합 선택"""
    if not all_results:
        print("분석할 결과가 없습니다.")
        return pd.DataFrame(), {}

    print("\n" + "="*80)
    print("결과 분석")
    print("="*80)

    # 결과를 DataFrame으로 변환
    results_data = []
    for key, result in all_results.items():
        metrics = result['metrics']
        row_data = {
            'combination': key,
            'model': result['model'],
            'normalization': result['normalization'],
            'normalization_name': result['normalization_name'],
            'prediction_count': result['prediction_count'],
            'training_time': result['training_time']
        }
        if metrics: # metrics 딕셔너리가 비어있지 않은 경우에만 추가
            row_data.update(metrics)
        results_data.append(row_data)

    results_df = pd.DataFrame(results_data)

    # MAPE 컬럼이 없는 경우를 대비하여 체크 (예측 대상에 실제 값이 없을 경우)
    if 'MAPE' not in results_df.columns or results_df['MAPE'].isna().all():
        print("경고: 'MAPE' 컬럼이 없어 성능 분석을 수행할 수 없습니다. 예측 대상 데이터에 실제 값이 있는지 확인하거나, 모든 모델에서 에러가 발생했을 수 있습니다.")
        return results_df, {}

    # 전체 결과 요약 (MAPE가 존재하는 경우)
    valid_results_df = results_df[results_df['MAPE'].notna()]
    if not valid_results_df.empty:
        print(f"총 {len(valid_results_df)}개 유효 조합 (메트릭스 계산 가능 기준)")
        print(f"평균 예측 수: {valid_results_df['prediction_count'].mean():.1f}개")
        print(f"최대 예측 수: {valid_results_df['prediction_count'].max()}개")
        print(f"평균 훈련 시간: {valid_results_df['training_time'].mean():.2f}초")
        print(f"평균 MAPE: {valid_results_df['MAPE'].mean():.2f}%")
    else:
        print("유효한 메트릭스 결과가 없어 전체 요약을 생성할 수 없습니다.")
        return results_df, {}


    # 모델별 최고 성능 찾기 (MAPE 기준)
    print("\n모델별 최고 성능 조합 (MAPE 기준 - 낮을수록 좋음):")
    print("-" * 50)

    model_categories = {
        '시계열': ['Prophet', 'SARIMA', 'ETS']
    }

    best_combinations = {}

    for category, models in model_categories.items():
        # 해당 카테고리 모델 중 MAPE가 존재하는 결과만 필터링
        category_results = results_df[results_df['model'].isin(models) & results_df['MAPE'].notna()]

        if not category_results.empty:
            # MAPE 기준 최고 성능 (MAPE는 낮을수록 좋음)
            best_row = category_results.loc[category_results['MAPE'].idxmin()]
            best_combinations[category] = best_row

            print(f"{category}: {best_row['model']} + {best_row['normalization_name']}")
            print(f"  MAPE: {best_row['MAPE']:.2f}%, 예측 수: {best_row['prediction_count']}개, 시간: {best_row['training_time']:.1f}초")
        else:
            print(f"{category}: 해당 카테고리에서 유효한 MAPE 결과가 없습니다.")


    # 전체 최고 성능 (MAPE 기준)
    if not valid_results_df.empty:
        overall_best = valid_results_df.loc[valid_results_df['MAPE'].idxmin()]
        print(f"\n전체 최고 성능 (MAPE 기준 - 낮을수록 좋음):")
        print(f"{overall_best['model']} + {overall_best['normalization_name']}")
        print(f"MAPE: {overall_best['MAPE']:.2f}%, 예측 수: {overall_best['prediction_count']}개, 시간: {overall_best['training_time']:.1f}초")
    else:
        print("\n전체 최고 성능을 찾을 수 없습니다 (유효한 메트릭스 결과 없음).")

    return results_df, best_combinations

# --- 8. 실험 실행 ---
print("사용자 선택 조건으로 학습 모델링을 시작합니다...")

# 실제 실험 실행
experiment_results = run_all_combinations_experiment()
results_df, best_combinations = analyze_results(experiment_results)

# 결과를 전역 변수로 저장 (향후 시각화 등에서 사용)
final_results_df = results_df.copy() if results_df is not None else pd.DataFrame()

if not final_results_df.empty:
    print(f"\n실험 완료: {len(final_results_df)}개 조합 결과 저장됨")
else:
    print("실험 결과가 없습니다.")

17:47:41 - cmdstanpy - INFO - Chain [1] start processing


사용자 선택 조건으로 학습 모델링을 시작합니다...
모델 학습 시작

정상화 방법 0: 코로나 기간 제외
------------------------------------------------------------
정상화 방법 0 적용 중...
필터링 결과:
  전체 데이터: 50996개
  훈련 데이터: 49776개 (2005-01 ~ 2024-12)
  예측 대상 (검증): 1220개 (2025-01 ~ 2025-05)

[1/18] Prophet + 정상화0
Prophet 모델 훈련 중...


17:47:41 - cmdstanpy - INFO - Chain [1] done processing
17:47:41 - cmdstanpy - INFO - Chain [1] start processing
17:47:41 - cmdstanpy - INFO - Chain [1] done processing
17:47:41 - cmdstanpy - INFO - Chain [1] start processing
17:47:41 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] start processing
17:47:42 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] start processing
17:47:42 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] start processing
17:47:42 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] start processing
17:47:42 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] start processing
17:47:42 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] start processing
17:47:42 - cmdstanpy - INFO - Chain [1] done processing
17:47:42 - cmdstanpy - INFO - Chain [1] 

17:47:52 - cmdstanpy - INFO - Chain [1] done processing
17:47:52 - cmdstanpy - INFO - Chain [1] start processing
17:47:52 - cmdstanpy - INFO - Chain [1] done processing
17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing
17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing
17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing
17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing
17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing
17:47:53 - cmdstanpy - INFO - Chain [1] start processing
17:47:53 - cmdstanpy - INFO - Chain [1] done processing
17:47:54 - cmdstanpy - INFO - Chain [1] start processing
17:47:54 - cmdstanpy - INFO - Chain [1] done processing
17:47:54 - cmdstanpy - INFO - Chain [1] 

17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:04 - cmdstanpy - INFO - Chain [1] start processing
17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:04 - cmdstanpy - INFO - Chain [1] start processing
17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:04 - cmdstanpy - INFO - Chain [1] start processing
17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:04 - cmdstanpy - INFO - Chain [1] start processing
17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:04 - cmdstanpy - INFO - Chain [1] start processing
17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:04 - cmdstanpy - INFO - Chain [1] start processing
17:48:04 - cmdstanpy - INFO - Chain [1] done processing
17:48:05 - cmdstanpy - INFO - Chain [1] start processing
17:48:05 - cmdstanpy - INFO - Chain [1] done processing
17:48:05 - cmdstanpy - INFO - Chain [1] start processing
17:48:05 - cmdstanpy - INFO - Chain [1] done processing
17:48:05 - cmdstanpy - INFO - Chain [1] 

...

18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:03 - cmdstanpy - INFO - Chain [1] start processing
18:00:03 - cmdstanpy - INFO - Chain [1] done processing
18:00:04 - cmdstanpy - INFO - Chain [1] start processing
18:00:04 - cmdstanpy - INFO - Chain [1] done processing
18:00:04 - cmdstanpy - INFO - Chain [1] start processing
18:00:04 - cmdstanpy - INFO - Chain [1]

18:00:14 - cmdstanpy - INFO - Chain [1] done processing
18:00:14 - cmdstanpy - INFO - Chain [1] start processing
18:00:14 - cmdstanpy - INFO - Chain [1] done processing
18:00:14 - cmdstanpy - INFO - Chain [1] start processing
18:00:14 - cmdstanpy - INFO - Chain [1] done processing
18:00:14 - cmdstanpy - INFO - Chain [1] start processing
18:00:15 - cmdstanpy - INFO - Chain [1] done processing
18:00:15 - cmdstanpy - INFO - Chain [1] start processing
18:00:15 - cmdstanpy - INFO - Chain [1] done processing
18:00:15 - cmdstanpy - INFO - Chain [1] start processing
18:00:15 - cmdstanpy - INFO - Chain [1] done processing
18:00:15 - cmdstanpy - INFO - Chain [1] start processing
18:00:15 - cmdstanpy - INFO - Chain [1] done processing
18:00:15 - cmdstanpy - INFO - Chain [1] start processing
18:00:15 - cmdstanpy - INFO - Chain [1] done processing
18:00:15 - cmdstanpy - INFO - Chain [1] start processing
18:00:15 - cmdstanpy - INFO - Chain [1] done processing
18:00:15 - cmdstanpy - INFO - Chain [1] 

18:00:25 - cmdstanpy - INFO - Chain [1] done processing
18:00:25 - cmdstanpy - INFO - Chain [1] start processing
18:00:25 - cmdstanpy - INFO - Chain [1] done processing
18:00:25 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:26 - cmdstanpy - INFO - Chain [1] start processing
18:00:26 - cmdstanpy - INFO - Chain [1] done processing
18:00:27 - cmdstanpy - INFO - Chain [1] 

18:00:37 - cmdstanpy - INFO - Chain [1] done processing
18:00:37 - cmdstanpy - INFO - Chain [1] start processing
18:00:37 - cmdstanpy - INFO - Chain [1] done processing
18:00:37 - cmdstanpy - INFO - Chain [1] start processing
18:00:37 - cmdstanpy - INFO - Chain [1] done processing
18:00:37 - cmdstanpy - INFO - Chain [1] start processing
18:00:37 - cmdstanpy - INFO - Chain [1] done processing
18:00:37 - cmdstanpy - INFO - Chain [1] start processing
18:00:37 - cmdstanpy - INFO - Chain [1] done processing
18:00:37 - cmdstanpy - INFO - Chain [1] start processing
18:00:37 - cmdstanpy - INFO - Chain [1] done processing
18:00:37 - cmdstanpy - INFO - Chain [1] start processing
18:00:38 - cmdstanpy - INFO - Chain [1] done processing
18:00:38 - cmdstanpy - INFO - Chain [1] start processing
18:00:38 - cmdstanpy - INFO - Chain [1] done processing
18:00:38 - cmdstanpy - INFO - Chain [1] start processing
18:00:38 - cmdstanpy - INFO - Chain [1] done processing
18:00:38 - cmdstanpy - INFO - Chain [1] 

Prophet 완료 - 예측 수: 1220개, 시간: 38.05초, MAPE: 558190804.93%
  완료: 예측 수 1220개, 시간 38.05초, MAPE: 558190804.93%

[11/18] SARIMA + 정상화3
SARIMA 모델 훈련 중...
계산 시간 단축을 위해 10개 그룹만 샘플링
